In [1]:
#1 Imports
import pandas as pd
from sklearn.model_selection import train_test_split

DRIVE_PATH = "/content/drive/MyDrive/Capstone/Data_2/"

In [2]:
#2 Loading the raw, merged, filtered NHANES data saved by G1
nhanes_core = pd.read_csv(DRIVE_PATH + "NHANES_core_raw.csv")
nhanes_full = pd.read_csv(DRIVE_PATH + "NHANES_full_raw.csv")

print("Core shape:", nhanes_core.shape)
print("Full shape:", nhanes_full.shape)

Core shape: (2401, 6)
Full shape: (2401, 8)


**Core feature set (Glucose, BloodPressure, BMI, Insulin, Age)**

In [3]:
#3 Separating features and target — Core
X_core = nhanes_core.drop("Outcome", axis=1)
y_core = nhanes_core["Outcome"]

In [4]:
#4 Train-Test Split — BEFORE any imputation
# Same leakage-prevention rationale as PIDD (Section 3.X.3): every
# imputation statistic must be learned from the training partition only.
X_train_core, X_test_core, y_train_core, y_test_core = train_test_split(
    X_core, y_core,
    test_size=0.2,
    random_state=42,
    stratify=y_core
)

print("Train shape:", X_train_core.shape)
print("Test shape :", X_test_core.shape)
print("\nTrain Outcome distribution:")
print(y_train_core.value_counts(normalize=True))

Train shape: (1920, 5)
Test shape : (481, 5)

Train Outcome distribution:
Outcome
0    0.792708
1    0.207292
Name: proportion, dtype: float64


In [5]:
#5 Median Imputation — all four affected variables
# Unlike PIDD, no NHANES variable exceeds ~13% missingness, well below the
# threshold that justified KNN imputation for Insulin/SkinThickness in PIDD
# (Section 3.X.4.2). Applying this study's missingness-based decision rule
# consistently, median imputation alone is appropriate here.
median_cols = ["Glucose", "BloodPressure", "BMI", "Insulin"]
median_values = X_train_core[median_cols].median()

print("Median values (from training set only):")
print(median_values)

X_train_core[median_cols] = X_train_core[median_cols].fillna(median_values)
X_test_core[median_cols] = X_test_core[median_cols].fillna(median_values)

Median values (from training set only):
Glucose          104.000
BloodPressure     72.000
BMI               28.500
Insulin            9.955
dtype: float64


In [6]:
#6 Final sanity check — confirm no missing values remain
print("Remaining nulls in X_train_core:")
print(X_train_core.isnull().sum())

print("\nRemaining nulls in X_test_core:")
print(X_test_core.isnull().sum())

Remaining nulls in X_train_core:
Glucose          0
BloodPressure    0
BMI              0
Insulin          0
Age              0
dtype: int64

Remaining nulls in X_test_core:
Glucose          0
BloodPressure    0
BMI              0
Insulin          0
Age              0
dtype: int64


In [7]:
#7 Saving the cleaned, split core dataset
X_train_core.to_csv(DRIVE_PATH + "NHANES_X_train_clean.csv", index=False)
X_test_core.to_csv(DRIVE_PATH + "NHANES_X_test_clean.csv", index=False)
y_train_core.to_csv(DRIVE_PATH + "NHANES_y_train_clean.csv", index=False)
y_test_core.to_csv(DRIVE_PATH + "NHANES_y_test_clean.csv", index=False)

print("Core dataset saved successfully")

Core dataset saved successfully


**Full feature set (core + Gender, Race)**

In [8]:
#8 Separating features and target — Full
X_full = nhanes_full.drop("Outcome", axis=1)
y_full = nhanes_full["Outcome"]

In [9]:
#9 Train-Test Split
# Same random_state and stratify target as the core split. Since nhanes_core
# and nhanes_full share identical row order (full = core + two extra columns,
# no rows added or removed), this produces the SAME train/test row split as
# the core dataset — important for later comparing core vs. full models on
# genuinely matched patients, not just matched sample sizes.
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full
)

print("Train shape:", X_train_full.shape)
print("Test shape :", X_test_full.shape)

Train shape: (1920, 7)
Test shape : (481, 7)


In [10]:
#10 Confirm Gender/Race have no missing values (checked in G1, verifying again here)
print(X_train_full[["Gender", "Race"]].isnull().sum())

Gender    0
Race      0
dtype: int64


In [11]:
#11 Median Imputation — same four numeric variables as core
# Gender and Race are untouched: both are complete (verified above) and, as
# categorical demographic fields, would require a different imputation
# strategy entirely if they ever did contain missing values.
median_cols = ["Glucose", "BloodPressure", "BMI", "Insulin"]
median_values_full = X_train_full[median_cols].median()

X_train_full[median_cols] = X_train_full[median_cols].fillna(median_values_full)
X_test_full[median_cols] = X_test_full[median_cols].fillna(median_values_full)

print("Remaining nulls in X_train_full:")
print(X_train_full.isnull().sum())

Remaining nulls in X_train_full:
Glucose          0
BloodPressure    0
BMI              0
Insulin          0
Age              0
Gender           0
Race             0
dtype: int64


In [12]:
#12 Saving the cleaned, split full dataset
X_train_full.to_csv(DRIVE_PATH + "NHANES_full_X_train_clean.csv", index=False)
X_test_full.to_csv(DRIVE_PATH + "NHANES_full_X_test_clean.csv", index=False)
y_train_full.to_csv(DRIVE_PATH + "NHANES_full_y_train_clean.csv", index=False)
y_test_full.to_csv(DRIVE_PATH + "NHANES_full_y_test_clean.csv", index=False)

print("Full dataset saved successfully")

Full dataset saved successfully
